In [ ]:
import pandas as pd

df = pd.read_csv("Training.csv")


symptom_cols = [col for col in df.columns if col != 'prognosis']


records = []
for _, row in df.iterrows():
    symptoms = [symptom for symptom in symptom_cols if row[symptom] == 1]
    records.append({
        "Disease": row["prognosis"],
        "Symptoms": ", ".join(symptoms)
    })

converted_df = pd.DataFrame(records)

converted_df.to_csv("normal_encoded_dataset.csv", index=False)

print("Conversion complete! Saved as normal_encoded_dataset.csv")
print(converted_df.head())


✅ Conversion complete! Saved as normal_encoded_dataset.csv
            Disease                                           Symptoms
0  Fungal infection  itching, skin_rash, nodal_skin_eruptions, disc...
1  Fungal infection  skin_rash, nodal_skin_eruptions, dischromic _p...
2  Fungal infection  itching, nodal_skin_eruptions, dischromic _pat...
3  Fungal infection            itching, skin_rash, dischromic _patches
4  Fungal infection           itching, skin_rash, nodal_skin_eruptions


In [ ]:
import pandas as pd

df = pd.read_csv("normal_encoded_dataset.csv")

df["Symptoms"] = df["Symptoms"].str.replace("_", " ").str.lower().str.strip()
df["Disease"] = df["Disease"].str.strip().str.lower()

df.head()


,Disease,Symptoms
0,fungal infection,"itching, skin rash, nodal skin eruptions, disc..."
1,fungal infection,"skin rash, nodal skin eruptions, dischromic p..."
2,fungal infection,"itching, nodal skin eruptions, dischromic pat..."
3,fungal infection,"itching, skin rash, dischromic patches"
4,fungal infection,"itching, skin rash, nodal skin eruptions"


In [130]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["Symptoms"])
y = df["Disease"]


In [131]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 1.0


In [132]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [133]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yash1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yash1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yash1\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [134]:
def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9_, ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    tokens = []
    for tok in re.split(r"[,\s]+", s):
        tok = tok.strip()
        if not tok:
            continue
        if tok in stop_words:
            continue
        tok = lemmatizer.lemmatize(tok)
        tokens.append(tok)
    return " ".join(tokens)


In [135]:
def combine_symptoms(df):
    symptom_cols = [c for c in df.columns if c.lower().startswith('symptom')]
    df_m = df.melt(id_vars=["Disease"], value_vars=symptom_cols, value_name="Symptom")
    df_m = df_m.dropna(subset=["Symptom"])
    df_m["Symptom"] = df_m["Symptom"].astype(str).str.strip()
    df_m = df_m[df_m["Symptom"] != ""]
    grouped = df_m.groupby('Disease')['Symptom'].unique().reset_index()
    grouped['Symptoms'] = grouped['Symptom'].apply(lambda x: ', '.join(sorted(set([s.strip() for s in x if s and isinstance(s, str)]))))
    return grouped[['Disease', 'Symptoms']]


In [136]:
def build_index(df_symptoms):
    df_symptoms['cleaned_text'] = df_symptoms['Symptoms'].apply(clean_text)
    vectorizer = TfidfVectorizer(ngram_range=(1,2))
    X = vectorizer.fit_transform(df_symptoms['cleaned_text'])
    return vectorizer, X


In [137]:
def predict_diseases(user_text, df_symptoms, vectorizer, X, top_k=3):
    user_clean = clean_text(user_text)
    if not user_clean:
        return "Please describe symptoms (e.g. 'itching and skin_rash')."
    v = vectorizer.transform([user_clean])
    sim = cosine_similarity(v, X)[0]
    idx = np.argsort(sim)[::-1][:top_k]
    results = []
    for i in idx:
        score = float(sim[i])
        if score <= 0:
            continue
        disease = df_symptoms.iloc[i]['Disease']
        symptoms = df_symptoms.iloc[i]['Symptoms']
        results.append((disease, symptoms, score))
    if not results:
        return "No matching disease found. Try listing more or different symptoms."
    out_lines = []
    for d, s, sc in results:
        out_lines.append(f"{d} (score: {sc:.3f})\nSymptoms: {s}")
    return "\n\n".join(out_lines)


In [138]:
def create_app(df):
    if "Symptoms" not in df.columns:
        raise ValueError("Expected a column named 'Symptoms' with combined symptoms.")
    df_symptoms = df.copy()
    df_symptoms["Symptoms"] = df_symptoms["Symptoms"].astype(str)
    df_symptoms["cleaned_text"] = df_symptoms["Symptoms"].apply(clean_text)
    df_symptoms = df_symptoms[df_symptoms["cleaned_text"].str.strip() != ""]
    vectorizer = TfidfVectorizer(ngram_range=(1,2))
    X = vectorizer.fit_transform(df_symptoms["cleaned_text"])

    def respond(text, top_k=3):
        return predict_diseases(text, df_symptoms, vectorizer, X, top_k=top_k)

    demo = gr.Interface(
        fn=respond,
        inputs=[
            gr.Textbox(lines=3, placeholder="Describe symptoms, e.g. 'itching and skin_rash'"),
            gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Top K matches")
        ],
        outputs=gr.Textbox(
            label="Predicted diseases (with matching score and known symptoms)",
            lines=10,    
            placeholder="Predicted diseases will appear here..."
        ),
        title="Symptom → Disease Assistant",
        description="Type symptoms and get top matching diseases from the dataset. (Educational use only)"
    )
    return demo, df_symptoms, vectorizer, X


In [139]:
try:
    df
except NameError:
    raise RuntimeError("Please provide a DataFrame named `df` or modify the script to read a CSV (see comments).")

demo, df_symptoms, vectorizer, X = create_app(df)
demo.launch()


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [140]:
df.head(20)

,Disease,Symptoms
0,fungal infection,"itching, skin rash, nodal skin eruptions, disc..."
1,fungal infection,"skin rash, nodal skin eruptions, dischromic p..."
2,fungal infection,"itching, nodal skin eruptions, dischromic pat..."
3,fungal infection,"itching, skin rash, dischromic patches"
4,fungal infection,"itching, skin rash, nodal skin eruptions"
5,fungal infection,"skin rash, nodal skin eruptions, dischromic p..."
6,fungal infection,"itching, nodal skin eruptions, dischromic pat..."
7,fungal infection,"itching, skin rash, dischromic patches"
8,fungal infection,"itching, skin rash, nodal skin eruptions"
9,fungal infection,"itching, skin rash, nodal skin eruptions, disc..."


In [141]:
df.to_csv("cleaned_symptoms_new.csv",index=False)

In [142]:
import pickle

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("df_symptoms.pkl", "wb") as f:
    pickle.dump(df_symptoms, f)

with open("tfidf_matrix.pkl", "wb") as f:
    pickle.dump(X, f)


In [143]:
import pickle

with open("tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("df_symptoms.pkl", "rb") as f:
    df_symptoms = pickle.load(f)

with open("tfidf_matrix.pkl", "rb") as f:
    X = pickle.load(f)


In [144]:
import gradio as gr
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def predict_diseases(user_text, df_symptoms, vectorizer, X, top_k=3):
    user_clean = clean_text(user_text)
    if not user_clean:
        return "Please describe symptoms (e.g. 'itching and skin_rash')."
    v = vectorizer.transform([user_clean])
    sim = cosine_similarity(v, X)[0]
    idx = np.argsort(sim)[::-1][:top_k]
    results = []
    for i in idx:
        score = float(sim[i])
        if score <= 0:
            continue
        disease = df_symptoms.iloc[i]['Disease']
        symptoms = df_symptoms.iloc[i]['Symptoms']
        results.append((disease, symptoms, score))
    if not results:
        return "No matching disease found. Try listing more or different symptoms."
    
    out_lines = []
    for d, s, sc in results:
        description = ""
        if 'dk' in globals() and d in dk['Disease'].values:
            description = dk[dk['Disease'] == d]['Description'].values[0]
        out_lines.append(f"{d} (score: {sc:.3f})\nSymptoms: {s}\nDescription: {description}")
    return "\n\n".join(out_lines)


def create_app(df):
    if "Symptoms" not in df.columns:
        raise ValueError("Expected a column named 'Symptoms' with combined symptoms.")
    df_symptoms = df.copy()
    df_symptoms["Symptoms"] = df_symptoms["Symptoms"].astype(str)
    df_symptoms["cleaned_text"] = df_symptoms["Symptoms"].apply(clean_text)
    df_symptoms = df_symptoms[df_symptoms["cleaned_text"].str.strip() != ""]
    vectorizer = TfidfVectorizer(ngram_range=(1,2))
    X = vectorizer.fit_transform(df_symptoms["cleaned_text"])

    def respond(text, top_k=3):
        return predict_diseases(text, df_symptoms, vectorizer, X, top_k=top_k)

    demo = gr.Interface(
        fn=respond,
        inputs=[
            gr.Textbox(lines=3, placeholder="Describe symptoms, e.g. 'itching and skin_rash'"),
            gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Top K matches")
        ],
        outputs=gr.Textbox(
            label="Predicted diseases (with matching score, symptoms, and description)",
            lines=15,
            placeholder="Predicted diseases will appear here..."
        ),
        title="Symptom → Disease Assistant",
        description="Type symptoms and get top matching diseases from the dataset. (Educational use only)"
    )
    return demo, df_symptoms, vectorizer, X


try:
    df
except NameError:
    raise RuntimeError("Please provide a DataFrame named `df`.")

demo, df_symptoms, vectorizer, X = create_app(df)
demo.launch()




* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [145]:
dk=pd.read_csv("E:\AI_medico\symptom_Description.csv")

In [146]:
dl=pd.read_csv("symptom_precaution.csv")

In [147]:
def predict_diseases(user_text, df_symptoms, vectorizer, X, top_k=3):
    user_clean = clean_text(user_text)
    if not user_clean:
        return "Please describe symptoms (e.g. 'itching and skin_rash')."

    # Vectorize input and compute cosine similarity
    v = vectorizer.transform([user_clean])
    sim = cosine_similarity(v, X)[0]

    # Get top K indices
    idx = np.argsort(sim)[::-1][:top_k]
    results = []

    for i in idx:
        if i >= len(df_symptoms):
            continue

        score = float(sim[i])
        if score <= 0:
            continue

        row = df_symptoms.iloc[i]
        disease = row.get('Disease', 'Unknown Disease')
        symptoms = row.get('Symptoms', 'No symptoms listed')

        #Match disease name in dk to get its description
        if 'Disease' in dk.columns and 'Description' in dk.columns:
            match = dk.loc[dk['Disease'].str.lower() == str(disease).lower(), 'Description']
            description = match.iloc[0] if not match.empty else "No description available."
        else:
            description = "No description available."

        results.append((disease, symptoms, description, score))

    if not results:
        return "No matching disease found. Try listing more or different symptoms."

    # Build output string
    out_lines = []
    for d, s, desc, sc in results:
        out_lines.append(
            f"{d} (Score: {sc:.3f})\n"
            f"Description: {desc}\n"
            f"Symptoms: {s}"
        )

    return "\n\n".join(out_lines)
# ---------------------------------------------------------

# Gradio app creation
def create_app(df):
    if "Symptoms" not in df.columns:
        raise ValueError("Expected a column named 'Symptoms' with combined symptoms.")

    df_symptoms = df.copy()
    df_symptoms["Symptoms"] = df_symptoms["Symptoms"].astype(str)
    df_symptoms["cleaned_text"] = df_symptoms["Symptoms"].apply(clean_text)
    df_symptoms = df_symptoms[df_symptoms["cleaned_text"].str.strip() != ""]

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    X = vectorizer.fit_transform(df_symptoms["cleaned_text"])

    def respond(text, top_k=3):
        return predict_diseases(text, df_symptoms, vectorizer, X, top_k=top_k)

    demo = gr.Interface(
        fn=respond,
        inputs=[
            gr.Textbox(lines=3, placeholder="Describe symptoms, e.g. 'itching and skin_rash'"),
            gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Top K matches")
        ],
        outputs=gr.Textbox(
            label="Predicted diseases (with score, description & known symptoms)",
            lines=10,
            placeholder="Predicted diseases will appear here..."
        ),
        title="🩺 Symptom → Disease Assistant",
        description="Enter your symptoms to find likely diseases with explanations. (Educational use only)"
    )
    return demo, df_symptoms, vectorizer, X
# ---------------------------------------------------------

# Create and launch app
demo, df_symptoms, vectorizer, X = create_app(df)
demo.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [148]:
df.head()

,Disease,Symptoms
0,fungal infection,"itching, skin rash, nodal skin eruptions, disc..."
1,fungal infection,"skin rash, nodal skin eruptions, dischromic p..."
2,fungal infection,"itching, nodal skin eruptions, dischromic pat..."
3,fungal infection,"itching, skin rash, dischromic patches"
4,fungal infection,"itching, skin rash, nodal skin eruptions"


In [149]:
dk.head()

,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi..."
4,Psoriasis,Psoriasis is a common skin disorder that forms...


In [150]:
dl.head()

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [151]:
precaution_cols = [col for col in df.columns if col.startswith("Precaution")]
df_melted = df.melt(id_vars=["Disease"], value_vars=precaution_cols, value_name="Precaution")
df_melted = df_melted.dropna(subset=["Precaution"])
df_melted["Precaution"] = df_melted["Precaution"].astype(str).str.strip()
df_combined = df_melted.groupby("Disease")["Precaution"].unique().reset_index()
df_combined["All_Precaution"] = df_combined["Precaution"].apply(lambda x: "  |  ".join(sorted(set(x))))
df_final = df_combined[["Disease", "All_Precaution"]]
print(df_final)


Empty DataFrame
Columns: [Disease, All_Precaution]
Index: []


In [152]:
import pandas as pd
dd = pd.read_csv("symptom_precaution.csv")
precaution_table = dd.melt(id_vars=["Disease"], value_vars=["Precaution_1","Precaution_2","Precaution_3","Precaution_4"], var_name="Precaution_Number", value_name="Precaution")
precaution_table = precaution_table.dropna(subset=["Precaution"]).drop(columns=["Precaution_Number"]).reset_index(drop=True)
print(precaution_table)

             Disease                        Precaution
0      Drug Reaction                   stop irritation
1            Malaria          Consult nearest hospital
2            Allergy                    apply calamine
3     Hypothyroidism                     reduce stress
4          Psoriasis  wash hands with warm soapy water
..               ...                               ...
157      Hepatitis D                         follow up
158        Pneumonia                         follow up
159        Arthritis                           massage
160  Gastroenteritis             ease back into eating
161     Tuberculosis                              rest

[162 rows x 2 columns]


In [153]:
dm=pd.DataFrame(precaution_table)

In [154]:
dm

,Disease,Precaution
0,Drug Reaction,stop irritation
1,Malaria,Consult nearest hospital
2,Allergy,apply calamine
3,Hypothyroidism,reduce stress
4,Psoriasis,wash hands with warm soapy water
...,...,...
157,Hepatitis D,follow up
158,Pneumonia,follow up
159,Arthritis,massage
160,Gastroenteritis,ease back into eating


In [155]:
dm = dm.groupby("Disease", as_index=False)["Precaution"].apply(lambda x: "  ".join(x)).reset_index(drop=True)
print(dm)

                                    Disease  \
0   (vertigo) Paroymsal  Positional Vertigo   
1                                      AIDS   
2                                      Acne   
3                       Alcoholic hepatitis   
4                                   Allergy   
5                                 Arthritis   
6                          Bronchial Asthma   
7                      Cervical spondylosis   
8                               Chicken pox   
9                       Chronic cholestasis   
10                              Common Cold   
11                                   Dengue   
12                                Diabetes    
13             Dimorphic hemmorhoids(piles)   
14                            Drug Reaction   
15                         Fungal infection   
16                                     GERD   
17                          Gastroenteritis   
18                             Heart attack   
19                              Hepatitis B   
20           

In [156]:
dm

,Disease,Precaution
0,(vertigo) Paroymsal Positional Vertigo,lie down avoid sudden change in body avoid a...
1,AIDS,avoid open cuts wear ppe if possible consult...
2,Acne,bath twice avoid fatty spicy food drink plen...
3,Alcoholic hepatitis,stop alcohol consumption consult doctor medi...
4,Allergy,apply calamine cover area with bandage use i...
5,Arthritis,exercise use hot and cold therapy try acupun...
6,Bronchial Asthma,switch to loose cloothing take deep breaths ...
7,Cervical spondylosis,use heating pad or cold pack exercise take o...
8,Chicken pox,use neem in bathing consume neem leaves tak...
9,Chronic cholestasis,cold baths anti itch medicine consult doctor...


In [157]:
dm.to_csv("cleaned_precautions.csv", index=False)

In [158]:
dm=pd.read_csv('cleaned_precautions.csv')

In [ ]:
def predict_diseases(user_text, df_symptoms, vectorizer, X, top_k=3):
    user_clean = clean_text(user_text)
    if not user_clean:
        return "Please describe symptoms (e.g. 'itching and skin_rash')."

    # Vectorize input and compute cosine similarity
    v = vectorizer.transform([user_clean])
    sim = cosine_similarity(v, X)[0]

    # Get top K indices
    idx = np.argsort(sim)[::-1][:top_k]
    results = []

    for i in idx:
        if i >= len(df_symptoms):
            continue

        score = float(sim[i])
        if score <= 0:
            continue

        row = df_symptoms.iloc[i]
        disease = row.get('Disease', 'Unknown Disease')
        symptoms = row.get('Symptoms', 'No symptoms listed')

        #  Match disease name in dk to get description
        if 'Disease' in dk.columns and 'Description' in dk.columns:
            match = dk.loc[dk['Disease'].str.lower() == str(disease).lower(), 'Description']
            description = match.iloc[0] if not match.empty else "No description available."
        else:
            description = "No description available."

        #  Match disease name in dm to get precautions
        if 'Disease' in dm.columns and 'Precaution' in dm.columns:
            p_match = dm.loc[dm['Disease'].str.lower() == str(disease).lower(), 'Precaution']
            precautions = p_match.iloc[0] if not p_match.empty else "No precautions available."
        else:
            precautions = "No precautions available."

        results.append((disease, symptoms, description, precautions, score))

    if not results:
        return "No matching disease found. Try listing more or different symptoms."

    # Build output string neatly
    out_lines = []
    for d, s, desc, pre, sc in results:
        out_lines.append(
            f"{d} (Score: {sc:.3f})\n"
            f"Description: {desc}\n"
            f"Symptoms: {s}\n"
            f"Precautions: {pre}"
        )

    return "\n\n".join(out_lines)



# Gradio app creation
def create_app(df):
    if "Symptoms" not in df.columns:
        raise ValueError("Expected a column named 'Symptoms' with combined symptoms.")

    df_symptoms = df.copy()
    df_symptoms["Symptoms"] = df_symptoms["Symptoms"].astype(str)
    df_symptoms["cleaned_text"] = df_symptoms["Symptoms"].apply(clean_text)
    df_symptoms = df_symptoms[df_symptoms["cleaned_text"].str.strip() != ""]

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    X = vectorizer.fit_transform(df_symptoms["cleaned_text"])

    def respond(text, top_k=3):
        return predict_diseases(text, df_symptoms, vectorizer, X, top_k=top_k)

    demo = gr.Interface(
        fn=respond,
        inputs=[
            gr.Textbox(lines=3, placeholder="Describe symptoms, e.g. 'itching and skin_rash'"),
            gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Top K matches")
        ],
        outputs=gr.Textbox(
            label="Predicted diseases (with score, description, symptoms & precautions)",
            lines=12,
            placeholder="Predicted diseases will appear here..."
        ),
        title="🩺 Symptom → Disease Assistant",
        description="Enter your symptoms to find likely diseases with explanations and precautions. (Educational use only)"
    )
    return demo, df_symptoms, vectorizer, X


# Create and launch app
demo, df_symptoms, vectorizer, X = create_app(df)
demo.launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [160]:
print(df.columns)


Index(['Disease', 'Symptoms'], dtype='object')


In [161]:
import pickle

with open("df_symptoms.pkl", "wb") as f:
    pickle.dump(df_symptoms, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("X.pkl", "wb") as f:
    pickle.dump(X, f)

In [162]:
import pickle

with open("df_symptoms.pkl", "rb") as f:
    df_symptoms = pickle.load(f)

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("X.pkl", "rb") as f:
    X = pickle.load(f)

In [ ]:
import pickle
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd



# Load pickled data

with open("df_symptoms.pkl", "rb") as f:
    df_symptoms = pickle.load(f)
with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
with open("X.pkl", "rb") as f:
    X = pickle.load(f)

# ---------------------------------------------------------

#  Load CSVs

dk = pd.read_csv(r"E:\AI_medico\symptom_Description.csv")
dm = pd.read_csv(r"E:\AI_medico\cleaned_precautions.csv")

# ---------------------------------------------------------

# ✅ Normalize all disease names

for df_temp in [df_symptoms, dk, dm]:
    if "Disease" in df_temp.columns:
        df_temp["Disease"] = df_temp["Disease"].astype(str).str.lower().str.strip()

# ✅ Combine multiple precautions for same disease into one line

if "Precaution" in dm.columns:
    dm_combined = (
        dm.groupby("Disease")["Precaution"]
        .apply(lambda x: "; ".join(sorted(set(x.dropna().astype(str)))))
        .reset_index()
    )
else:
    dm_combined = pd.DataFrame(columns=["Disease", "Precaution"])

# ---------------------------------------------------------

# ✅ Helper: clean user input

def clean_text(text):
    return " ".join(text.lower().strip().split())

# ---------------------------------------------------------

# ✅ Disease prediction logic

def predict_diseases(user_text, df_symptoms, vectorizer, X, top_k=3):
    user_clean = clean_text(user_text)
    if not user_clean:
        return "Please describe symptoms (e.g. 'itching and skin_rash')."

    # Compute cosine similarity
    v = vectorizer.transform([user_clean])
    sim = cosine_similarity(v, X)[0]

    # Sort indices by similarity (descending)
    idx_sorted = np.argsort(sim)[::-1]

    results = []
    seen_diseases = set()

    for i in idx_sorted:
        if len(results) >= top_k:
            break
        if i >= len(df_symptoms):
            continue

        disease = str(df_symptoms.iloc[i].get("Disease", "")).strip().lower()
        if not disease or disease in seen_diseases:
            continue

        seen_diseases.add(disease)
        score = float(sim[i])
        if score <= 0:
            continue

        symptoms = df_symptoms.iloc[i].get("Symptoms", "No symptoms listed")

        # 🔹 Description lookup
        desc_row = dk.loc[dk["Disease"] == disease, "Description"]
        description = desc_row.iloc[0] if not desc_row.empty else "No description available."

        # 🔹 Precautions lookup
        pre_row = dm_combined.loc[dm_combined["Disease"] == disease, "Precaution"]
        precautions_text = pre_row.iloc[0] if not pre_row.empty else "No precautions available."

        results.append((disease, symptoms, description, precautions_text, score))

    if not results:
        return "No matching disease found. Try listing more or different symptoms."

    # ✅ Format output neatly
    out_lines = []
    for d, s, desc, pre, sc in results:
        out_lines.append(
            f"Disease: {d.title()} (Score: {sc:.3f})\n"
            f"Description: {desc}\n"
            f"Symptoms: {s}\n"
            f"Precautions: {pre}"
        )

    return "\n\n".join(out_lines)

# ---------------------------------------------------------

# ✅ Gradio Interface

def respond(text, top_k=3):
    return predict_diseases(text, df_symptoms, vectorizer, X, top_k=top_k)

demo = gr.Interface(
    fn=respond,
    inputs=[
        gr.Textbox(lines=3, placeholder="Describe symptoms, e.g. 'itching and skin_rash'"),
        gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Top K matches")
    ],
    outputs=gr.Textbox(
        label="Predicted diseases (with score, description, symptoms & precautions)",
        lines=14,
        placeholder="Predicted diseases will appear here..."
    ),
    title="🩺 Symptom → Disease Assistant",
    description="Enter your symptoms to find likely diseases with descriptions and precautions. (Educational use only)"
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
